# STRATA contra llama.cpp (Q3_K_M a Q8_0) numa T4

Antes de rodar: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Depois *Executar tudo*.

Compila só o `llama-quantize`, converte `unsloth/Llama-3.2-1B` para GGUF BF16, gera os k-quants e roda `tests/ppl/strata_ppl.py` no wikitext-2 inteiro com tudo na mesma tabela: BF16, estratos 1 a 4 do STRATA, INT8 progressivo e os GGUFs. Os GGUFs são desquantizados pelo `gguf-py` e colocados nas mesmas 112 matrizes que o STRATA corta, então a única diferença entre as linhas é o formato dos pesos.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = "claude/project-thread-qm128x"
!rm -rf strata && git clone -q --depth 1 -b $BRANCH https://github.com/Cantyll/strata.git
!rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!cd llama.cpp && git log -1 --format='%h %cd'

In [ ]:
!pip install -q -U transformers datasets safetensors
!pip install -q -e llama.cpp/gguf-py
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /dev/null && cmake --build build -j --target llama-quantize > /dev/null && ls build/bin

Converter para GGUF BF16 e quantizar. Sem imatrix: é o `llama-quantize` padrão.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download("unsloth/Llama-3.2-1B", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
!mkdir -p /content/gguf && python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile /content/gguf/bf16.gguf 2>&1 | tail -1
QUANTS = ["Q3_K_M", "Q4_K_S", "Q4_K_M", "Q5_K_S", "Q5_K_M", "Q6_K", "Q8_0"]
for q in QUANTS:
    !llama.cpp/build/bin/llama-quantize /content/gguf/bf16.gguf /content/gguf/{q}.gguf {q} > /content/gguf/{q}.log 2>&1
!ls -la /content/gguf

Tabela completa (cerca de 2 a 3 minutos por linha numa T4).

In [ ]:
%cd /content/strata
GG = ",".join(f"{q}=/content/gguf/{q}.gguf" for q in QUANTS)
!python tests/ppl/strata_ppl.py --model unsloth/Llama-3.2-1B --depths 1,2,3,4 --int-bits '' --prog-bits 5,6,7,8 --gguf {GG} --tag kquants --out /content/res
from IPython.display import Markdown, display
display(Markdown(open('/content/res/unsloth__Llama-3.2-1B_kquants.md').read()))

In [ ]:
from google.colab import files
files.download('/content/res/unsloth__Llama-3.2-1B_kquants.md')
files.download('/content/res/unsloth__Llama-3.2-1B_kquants.json')